# 《PythAPCS123》單元 13-5：語意錯誤（Logic Error）與常見邏輯盲點排查（WA 防範）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-5_logic_errors_and_wa_pitfalls.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：徹底克服競技程式中最讓人沮喪的「無聲殺手」——語意錯誤（Logic Error / Semantic Error）！當程式碼文法正確、執行過程沒有任何拋出崩潰（RE），但上傳至評判系統卻頻繁拿下刺眼的 Wrong Answer（WA）時，問題往往出在隱藏的邏輯死角。本單元深入剖析五大高頻邏輯暗坑：差一錯誤（Off-by-one，含閉區間個數與二分搜尋邊界）、運算子優先級失控（位元運算與負數次方陷阱）、二進位浮點數精度截斷與負數整數除法向負無窮取整、二維串列淺拷貝幽靈修改與可變預設參數陷阱、以及變數命名遮蔽內建函式與多測資未重置狀態污染。全面築起抵禦 WA 的鋼鐵防線！


### 13.5.1 什麼是語意錯誤？語法合法但運算邏輯背離題意的本質

在程式設計的世界中，如果說語法錯誤（SyntaxError）就像是「拼錯英文單字或漏打標點」，執行錯誤（Runtime Error）像是「開車開到懸崖邊翻車」，那麼**語意錯誤（Logic Error / Semantic Error）**就如同「你非常流暢地開著一輛性能完美的車，但導航目的地完全設定錯了方向」。

語意錯誤之所以被稱為競賽中最棘手的「隱形殺手」，原因在於它具有高度的隱蔽性：
1. **編譯器與直譯器完全沉默**：因為所有的單字、縮排、括號與型態完全符合 Python 規範，直譯器絕對不會主動跳出任何一行紅字警告。
2. **本機執行看似順利**：你隨手測了一筆最簡單的樣例，可能「剛好」碰巧算出看似正常的數字，但一旦遇到多樣化的隱藏測試點，程式就會原形畢露，給出完全錯誤的答案，被裁判系統無情判定為 **WA（Wrong Answer）**。

考場最典型的語意邏輯盲點：
- **條件式複製貼上手滑**：例如 `if x > max_val: min_val = x`（變數名稱對錯）。
- **迴圈內部提早 return**：初學者在寫搜尋函式時，把 `return False` 寫在迴圈內部的 `else` 分支裡，導致只要第 1 個元素不匹配就立刻宣判失敗退出！
- **邏輯連接詞 `and` vs `or` 混淆**：題目要求「大於 10 且小於 20」，手滑寫成「大於 10 或小於 20」。


In [ ]:
# 13.5.1 程式碼演示：三大典型語意錯誤剖析（提早 return、條件連接詞、變數錯位）
print("--- 語意陷阱 1: 迴圈內部提早 return 悲劇 ---")
# 需求：檢查串列中是否存在偶數

def faulty_has_even(nums):
    for x in nums:
        if x % 2 == 0:
            return True
        else:
            # 致命邏輯傷：只要第 1 個不是偶數，就直接 return False 收工！
            return False

def correct_has_even(nums):
    for x in nums:
        if x % 2 == 0:
            return True
    # 遍歷完所有元素確認都沒有，才能下結論
    return False

test_data = [1, 3, 5, 8] # 第 4 個元素是偶數
print(f"測試串列: {test_data}")
print("錯誤函式判定:", faulty_has_even(test_data), "❌ (看到 1 不是偶數就直接判 False，後續全沒看！)")
print("正確函式判定:", correct_has_even(test_data), "✅ (看完所有元素後正確回傳 True)")

print("\n--- 語意陷阱 2: and 與 or 混淆 ---")
# 需求：篩選大於 10 且小於 20 的數值
nums = [5, 12, 18, 25]
bad_filtered = [x for x in nums if x > 10 or x < 20] # 恆真句！全體皆入選
good_filtered = [x for x in nums if 10 < x < 20]     # 嚴謹交集

print("原陣列:", nums)
print("誤用 or 篩選結果:", bad_filtered, "❌ (所有數字都符合，WA！)")
print("正確交集篩選結果:", good_filtered, "✅ (僅 12, 18)")


### 13.5.1 語法重點回顧與核心觀念提煉

語意錯誤的診斷與防護心法：
1. **「遍歷搜尋」返回原則**：
   - 尋找「存在（Exist）」：命中條件時立刻 `return True`；若未命中「繼續迴圈」，直到「迴圈全數跑完後」才在最外層 `return False`。
   - 尋找「全體皆符合（For All）」：一旦出現反例立刻 `return False`；全數跑完確認無反例才 `return True`。
2. **手動追蹤小規模資料（Trace Table）**：在遇到結果不合預期時，取 3 到 5 個數字，拿張紙筆逐行手動模擬變數變化，是抓出邏輯偏差最有效的手段。
3. **題意逐字覆核**：特別留意題目中的「大於（`>`）」vs「大於等於（`>=`）」、「包含」vs「不包含」、「且」vs「或」。


In [ ]:
# 13.5.1 學生實作練習：嚴謹全體及格判定器
# 任務說明：實作 are_all_passing(scores) 函式
# 題意要求：檢查串列 scores 中的每一位學生成績是否「全數大於等於 60 分」
# 1. 若所有學生成績皆 >= 60，回傳 True
# 2. 只要有任何一位學生 < 60，回傳 False
# 3. 邊界特例：若班級為空串列 []，依常理回傳 True（Vacuous Truth）
# 請小心排查 return 的位置，絕不可在第一輪迴圈就草率結束！

def are_all_passing(scores: list) -> bool:
    # 請在此處實作嚴謹的搜尋邏輯
    for s in scores:
        if s < 60:
            return False
    return True

# 測試用例
print("全體及格:", are_all_passing([70, 85, 60, 92]))
print("一人不及格:", are_all_passing([80, 59, 90]))
print("空清單邊界:", are_all_passing([]))


In [ ]:
# 13.5.1 單元測試驗證
assert are_all_passing([60, 70, 80]) == True
assert are_all_passing([60, 59, 80]) == False
assert are_all_passing([50, 100]) == False
assert are_all_passing([100]) == True
assert are_all_passing([]) == True
print("🎉 13.5.1 所有測試通過！成功建立嚴謹的遍歷與邏輯判定思維！")


### 13.5.2 邏輯盲點一：差一錯誤（Off-by-one Error）

在電腦科學中，有一句著名的名言：「電腦科學領域只有兩大難題：快取失效、命名，以及**差一錯誤（Off-by-one Error, OBOE）**。」

差一錯誤指的是：你的演算法架構完全正確，但在迴圈計數、範圍設定、開閉區間或索引對齊時，剛好「多算了一次」或「少算了一次」。

考場四大經典差一地雷場景：
1. **`range(start, stop)` 的「左閉右開」特性**：
   Python 的 `range(a, b)` 永遠「包含起點 $a$，但不包含終點 $b$」（數學符號 $[a, b)$）。初學者若想從 $1$ 跑到 $10$，直覺寫下 `range(1, 10)`，迴圈只會跑到 $9$ 就收工，導致最後一筆資料被無辜丟失。
2. **閉區間 $[a, b]$ 的整數個數公式**：
   計算從 $a$ 到 $b$ 共有幾個整數時，初學者常直覺寫 `b - a`。例如從 3 號到 7 號，直接相減 $7 - 3 = 4$，但實際上包含 $3, 4, 5, 6, 7$ 共 **5 個整數**！正解公式恆為：**$b - a + 1$**！
3. **二分搜尋法區間收斂邊界**：
   在手刻二分搜尋時，條件必須寫 `while left <= right:`。若誤寫為 `while left < right:`，當搜尋範圍縮減到僅剩單一元素（`left == right`）且該元素剛好就是目標時，迴圈會提早終止，導致判定目標不存在！
4. **切片長度不變量**：對於 `s[start:end]`，切出來的子字串長度必定恆等於 `end - start`。


In [ ]:
# 13.5.2 程式碼演示：差一錯誤三重奏（range 上界、區間個數、二分搜尋）
print("--- 差一陷阱 1: range 左閉右開之累加缺失 ---")
# 需求：計算 1 到 10 的總和（公式 n*(n+1)//2 = 55）
faulty_sum = sum(range(1, 10))   # 只跑到 9
correct_sum = sum(range(1, 11))  # 跑到 10
print(f"誤寫 range(1, 10) 結果: {faulty_sum} ❌ (少了 10)")
print(f"正確 range(1, 11) 結果: {correct_sum} ✅")

print("\n--- 差一陷阱 2: 閉區間端點個數算錯 ---")
# 題目：計算從 3 號到 7 號選手共有幾人
start_id, end_id = 3, 7
wrong_count = end_id - start_id         # 7 - 3 = 4 (WA!)
correct_count = end_id - start_id + 1   # 7 - 3 + 1 = 5 (AC!)
print(f"區間 [{start_id}, {end_id}] 直接相減: {wrong_count} 人 ❌")
print(f"區間 [{start_id}, {end_id}] 加一公式: {correct_count} 人 ✅")

print("\n--- 差一陷阱 3: 二分搜尋 while left < right 漏判單點 ---")
def buggy_binary_search(arr, target):
    left = 0
    right = len(arr) - 1
    # 差一錯誤：漏掉等號，當 left == right 時無法檢查最後一個元素！
    while left < right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

test_arr = [10, 20, 30]
print("搜尋 30 (位於索引 2):", buggy_binary_search(test_arr, 30), "❌ (回傳 -1 找不到！)")


### 13.5.2 語法重點回顧與核心觀念提煉

差一錯誤的四步防禦口訣：
1. **起點檢驗**：迴圈的第 1 次迭代，變數的值是多少？是否吻合題目的起始條件？
2. **終點檢驗**：迴圈的最後 1 次迭代，變數的值是多少？是否精準抵達期望的終點邊界？
3. **區間計數公理**：對於整數閉區間 $[a, b]$，元素總個數恆為 **$b - a + 1$**。
4. **二分搜尋雙指標鐵律**：手刻二分搜尋一律寫 `while left <= right:`，縮減區間一律寫 `left = mid + 1` 與 `right = mid - 1`！


In [ ]:
# 13.5.2 學生實作練習：健全二分搜尋實作
# 任務說明：實作 accurate_binary_search(arr, target) 函式
# 在已排序串列 arr 中使用二分搜尋查找 target
# 嚴格要求：
# 1. 確保 while left <= right 邊界完整，絕不漏判 left == right 的端點情況！
# 2. 若找到回傳對應索引；若找不到安全回傳 -1

def accurate_binary_search(arr: list, target: int) -> int:
    left = 0
    right = len(arr) - 1
    # 請在此處實作完整的二分搜尋
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1

# 測試用例
sample = [5, 15, 25, 35, 45]
print("搜尋最左側 5:", accurate_binary_search(sample, 5))
print("搜尋最右側 45:", accurate_binary_search(sample, 45))
print("搜尋中間 25:", accurate_binary_search(sample, 25))
print("搜尋不存在的 99:", accurate_binary_search(sample, 99))


In [ ]:
# 13.5.2 單元測試驗證
assert accurate_binary_search([10], 10) == 0
assert accurate_binary_search([10], 20) == -1
assert accurate_binary_search([1, 2, 3, 4, 5], 1) == 0
assert accurate_binary_search([1, 2, 3, 4, 5], 5) == 4
assert accurate_binary_search([1, 2, 3, 4, 5], 3) == 2
assert accurate_binary_search([], 5) == -1
print("🎉 13.5.2 所有測試通過！成功徹底掃除差一錯誤（OBOE）！")


### 13.5.3 邏輯盲點二：運算優先級陷阱

當你在同一行程式碼中混合使用了算術運算、位元運算、比較運算以及邏輯運算時，直譯器會依照嚴格的**「運算子優先級（Operator Precedence）」**排定執行順序。如果忽視了優先級，程式算出來的結果將完全失控！

考場最高頻的四大優先級災難：
1. **加法與整數除法：`a + b // 2` 陷阱**：
   在計算兩數平均時，初學者常寫下 `mid = a + b // 2`。因為除法 `//` 優先於加法 `+`，電腦實際執行的是 $a + \frac{b}{2}$，而非 $\frac{a + b}{2}$！正確寫法必須加括號：`mid = (a + b) // 2`。
2. **位元運算與比較運算：`x & 1 == 0` 陷阱**：
   比較運算子 `==` 享有比位元運算子 `&` 更高的優先權！因此 `x & 1 == 0` 會先計算 `1 == 0`（結果為 `False`，等價於整數 0），再計算 `x & 0`，永遠得到 0！正確寫法「必須加括號」：`(x & 1) == 0`。
3. **負號與次方運算：`-2 ** 2` 陷阱**：
   在 Python 中，次方 `**` 優先於負號 `-`！因此 `-2 ** 2` 實際上是 `-(2 ** 2) = -4`，而不是 `(-2) ** 2 = 4`！
4. **邏輯 `not`、`and`、`or` 混用**：
   Python 的優先順序為：`not` 最優先，其次是 `and`，最後才是 `or`。

記住競賽防身第一金律：**「有任何疑慮，直接加上括號 `()`，括號是免費的，也是絕對清晰的保證！」**


In [ ]:
# 13.5.3 程式碼演示：優先級失控四大典型災難
print("--- 陷阱 1: 二分中點除法優先級 ---")
L, R = 10, 20
wrong_mid = L + R // 2   # 10 + (20 // 2) = 20
correct_mid = (L + R) // 2 # (10 + 20) // 2 = 15
print(f"中點計算 [10, 20]: 漏括號={wrong_mid} ❌ vs 加括號={correct_mid} ✅")

print("\n--- 陷阱 2: 位元運算與比較運算子 ---")
# 需求：檢查 4 是否為偶數
x = 4
# 錯誤寫法：Python 解讀為 4 & (1 == 0) -> 4 & False -> 0，在 if 判定中 0 視為 False！
wrong_even = (x & 1 == 0) 
# 正確寫法：先做位元運算
correct_even = ((x & 1) == 0)
print(f"4 是否為偶數: 漏括號 x & 1 == 0 評估為 {wrong_even} ❌")
print(f"4 是否為偶數: 加括號 (x & 1) == 0 評估為 {correct_even} ✅")

print("\n--- 陷阱 3: 負數次方運算 ---")
wrong_power = -3 ** 2     # -(3**2) = -9
correct_power = (-3) ** 2 # (-3)*(-3) = 9
print(f"-3 的平方: 未加括號 -3 ** 2 = {wrong_power} ❌")
print(f"-3 的平方: 加上括號 (-3) ** 2 = {correct_power} ✅")


### 13.5.3 語法重點回顧與核心觀念提煉

運算優先級記憶口訣與防身指南：
1. **括號小宇宙優先**：小括號 `()` 享有全語言最高優先級，能隨時打破任何預設順序。
2. **優先級天梯圖**：
   - 次方 `**`（最高算術）
   - 正負號 `+x`, `-x`
   - 乘除模 `*`, `/`, `//`, `%`
   - 加減 `+`, `-`
   - 位元運算 `&`, `^`, `|`
   - 比較運算 `==`, `!=`, `<`, `<=`
   - 邏輯運算 `not` -> `and` -> `or`
3. **競賽鐵律**：不要為了省兩下鍵盤而省略括號！多加括號完全不耗費執行效能，卻能保證邏輯 100% 按照預期運轉！


In [ ]:
# 13.5.3 學生實作練習：位元奇偶判斷與加權平均安全算式
# 任務說明：實作 calculate_weighted_and_parity(a, b, weight_a) 函式
# 1. 判斷整數 a 是否為奇數（利用位元運算 & 1），若為奇數 is_odd 設為 True，偶數設為 False
# 2. 計算加權公式：(a * weight_a + b * (100 - weight_a)) // 100
# 3. 回傳 (is_odd, weighted_result) 元組
# 務必加上明確括號，杜絕一切優先級造成的計算偏差！

def calculate_weighted_and_parity(a: int, b: int, weight_a: int) -> tuple:
    # 請在此處使用嚴格括號實作
    is_odd = ((a & 1) == 1)
    weighted_result = (a * weight_a + b * (100 - weight_a)) // 100
    return (is_odd, weighted_result)

# 測試用例
print("測試 (7, 10, 50):", calculate_weighted_and_parity(7, 10, 50))
print("測試 (4, 20, 80):", calculate_weighted_and_parity(4, 20, 80))


In [ ]:
# 13.5.3 單元測試驗證
assert calculate_weighted_and_parity(7, 10, 50) == (True, 8)
assert calculate_weighted_and_parity(4, 20, 80) == (False, 7)
assert calculate_weighted_and_parity(0, 100, 100) == (False, 0)
assert calculate_weighted_and_parity(11, 0, 100) == (True, 11)
print("🎉 13.5.3 所有測試通過！成功建立小括號絕對防禦戰略！")


### 13.5.4 邏輯盲點三：浮點數精確度與負數整數除法陷阱

在數值演算法題中，有兩種極易引爆 WA 的數值計算陷阱：

#### 陷阱一：二進位浮點數精度丟失（`0.1 + 0.2 != 0.3`）
電腦使用二進位儲存浮點數，十進位的 $0.1$ 與 $0.2$ 在二進位中是無限循環小數。相加結果為 `0.30000000000000004`。
- 若直接用 `a == b` 比對浮點數，幾乎必定回傳 `False`！
- *防禦對策*：比較兩浮點數時，一律判斷差值絕對值是否小於極小容差 $\epsilon$：`abs(a - b) < 1e-9`。

#### 陷阱二：負數整數除法「向負無窮取整」陷阱（Floor Division）
這是無數 C/C++ 轉 Python 選手在 APCS 考場痛失分數的超級暗坑！
- 在 C/C++ / Java 中，整數除法是**「向零截斷（Truncate towards zero）」**：`-7 / 2` 等於 `-3`。
- **在 Python 中，雙斜線 `//` 是「向下取整（Floor towards $-\infty$）」**：
  `-7 // 2` 的計算結果不是 `-3`，而是 **`-4`**！
  因為 $-3.5$ 向負無窮大方向取整就是 $-4$！
  若題目要求的是數理上的向零捨去（例如座標網格平移），直接使用 `//` 會讓負數結果全部差一！
  - *正解*：若需要向零截斷，必須寫 `int(-7 / 2)` 或 `-(abs(x) // d)`！


In [ ]:
# 13.5.4 程式碼演示：浮點精度容差比對 vs 負數整數除法陷阱
import math

print("--- 陷阱 1: 浮點數直接比大小之災難 ---")
a, b = 0.1, 0.2
print(f"0.1 + 0.2 的真實記憶體數值: {repr(a + b)}")
print(f"直接使用 == 0.3 比對: {(a + b) == 0.3} ❌ (WA 殺手！)")

# 防禦對策：容差比對
EPSILON = 1e-9
print(f"容差 abs((a+b) - 0.3) < 1e-9 比對: {abs((a + b) - 0.3) < EPSILON} ✅")

print("\n--- 陷阱 2: 負數整數除法向負無窮取整陷阱 ---")
# 計算 -7 除以 2
py_floor_div = -7 // 2        # Python 預設向下取整 -> -4
c_style_trunc = int(-7 / 2)   # 向零截斷 -> -3

print(f"Python 雙斜線 -7 // 2 結果: {py_floor_div} (向負無窮取整)")
print(f"向零截斷 int(-7 / 2) 結果: {c_style_trunc} (向零截斷)")
print("注意：如果題目要求求商向零靠攏，對負數直接用 // 會導致所有負數商數全數差 1！")

print("\n--- 陷阱 3: 先除後乘導致精度提早遺失 ---")
# 需求：計算 10 * 5 / 2
# 錯誤順序：先除後乘
loss_calc = (10 // 3) * 3 # (3) * 3 = 9 (提早整除丟失精度！)
# 正確順序：先乘後除
keep_calc = (10 * 3) // 3 # (30) // 3 = 10 (完全精確！)
print(f"先除後乘 (10 // 3) * 3 = {loss_calc} ❌ (丟失精度)")
print(f"先乘後除 (10 * 3) // 3 = {keep_calc} ✅ (完全無損)")


### 13.5.4 語法重點回顧與核心觀念提煉

數值運算防身三大心法：
1. **浮點比較必用容差**：`abs(a - b) < 1e-9`。
2. **負數除法認清方向**：
   - 需要地板除（Floor）：用 `a // b`。
   - 需要向零截斷（Truncate）：用 `int(a / b)` 或手動調整正負號。
3. **算術式「先乘後除」**：在進行整數比例計算時，永遠「先做分子相乘、最後才做分母整除」，防止中途因整數去尾刀而丟失精度！


In [ ]:
# 13.5.4 學生實作練習：向零截斷安全整數除法器
# 任務說明：實作 truncate_divide(dividend, divisor) 函式
# 實現類似 C 語言風格的「向零截斷（Truncate towards Zero）」整數除法
# 例如：
# truncate_divide(7, 2) == 3
# truncate_divide(-7, 2) == -3 （注意：不是 Python 預設的 -4！）
# truncate_divide(7, -2) == -3
# truncate_divide(-7, -2) == 3

def truncate_divide(dividend: int, divisor: int) -> int:
    # 請在此處實作向零截斷除法
    return int(dividend / divisor)

# 測試用例
print("7 // 2 向零截斷:", truncate_divide(7, 2))
print("-7 // 2 向零截斷:", truncate_divide(-7, 2))
print("-7 // -2 向零截斷:", truncate_divide(-7, -2))


In [ ]:
# 13.5.4 單元測試驗證
assert truncate_divide(7, 2) == 3
assert truncate_divide(-7, 2) == -3
assert truncate_divide(7, -2) == -3
assert truncate_divide(-7, -2) == 3
assert truncate_divide(0, 5) == 0
assert truncate_divide(5, 5) == 1
print("🎉 13.5.4 所有測試通過！徹底征服浮點精度與負數除法暗坑！")


### 13.5.5 邏輯盲點四：淺拷貝共享記憶體幽靈與可變預設參數陷阱

在 Python 中，串列（`list`）與字典（`dict`）屬於「可變物件（Mutable Objects）」。變數名稱實際上只是指向該記憶體區塊的「參照指針（Reference）」。忽視這點，會引爆兩大不可思議的幽靈臭蟲：

#### 幽靈一：二維網格乘號複製悲劇（`[[0]*W]*H`）
初學者為了省事寫出 `grid = [[0]*3]*3`。外層的 `*3` 並沒有建立 3 個獨立串列，而是將同一個串列物件複製了 3 份指針！修改 `grid[0][0] = 9`，會導致第 1、2、3 列的第 0 格全部同時變成 9！
- *正解*：強制使用列表生成式：`grid = [[0]*W for _ in range(H)]`。

#### 幽靈二：可變物件預設參數跨呼叫殘留陷阱
在定義函式時，初學者常寫出：
```python
# 致命陷阱：絕對不要用可變物件（如 []）作為函式預設參數！
def add_item(item, history=[]):
    history.append(item)
    return history
```
Python 在函式「定義期（Def Time）」只會建立這份 `[]` 串列一次！當函式被連續呼叫兩次時，第二次呼叫會直接沿用第一次呼叫殘留的 `history`，造成跨測資的嚴重狀態污染！
- *正解*：預設參數改用 `None`：
  ```python
  def add_item(item, history=None):
      if history is None:
          history = []
  ```


In [ ]:
# 13.5.5 程式碼演示：淺拷貝連動修改與可變預設參數跨呼叫污染
print("--- 幽靈 1: [[0]*3]*3 記憶體共享慘劇 ---")
bad_grid = [[0] * 3] * 3
bad_grid[0][0] = 9
print("修改 bad_grid[0][0] = 9 後，整張地圖全貌:")
for row in bad_grid:
    print(" ", row)
print("驚悚發現：每一列都被連動竄改！id 相等:", id(bad_grid[0]) == id(bad_grid[1]))

# 正確寫法：生成式建立獨立各列
good_grid = [[0] * 3 for _ in range(3)]
good_grid[0][0] = 9
print("\n使用生成式修正後:")
for row in good_grid:
    print(" ", row)
print("驗證：僅有第 0 列第 0 格為 9，其餘各列完全獨立！")

print("\n--- 幽靈 2: 可變預設參數跨呼叫污染 ---")
def faulty_append(val, container=[]):
    container.append(val)
    return container

# 第一次呼叫
res1 = faulty_append("A")
print("第一次呼叫 ('A'):", res1)

# 第二次呼叫（期望是新的空串列只裝 'B'）
res2 = faulty_append("B")
print("第二次呼叫 ('B'):", res2, "❌ (恐怖！前一次的 'A' 竟然幽靈殘留了！)")


### 13.5.5 語法重點回顧與核心觀念提煉

可變物件防禦三大心法：
1. **多維陣列外層必用生成式**：`[[0]*W for _ in range(H)]` 是考場永遠的標準解法。
2. **預設參數首選 `None`**：
   ```python
   def func(arr=None):
       if arr is None:
           arr = []
   ```
3. **複製串列切斷參照**：
   - 一維串列獨立複製：`b = a.copy()` 或 `b = a[:]`。
   - 二維陣列獨立複製：`import copy; b = copy.deepcopy(a)`。


In [ ]:
# 13.5.5 學生實作練習：安全歷史紀錄收集器
# 任務說明：實作 append_to_record_safely(entry, record_list=None) 函式
# 傳入 entry（任意項目）與 record_list（預設為 None）
# 需求：
# 1. 若呼叫端未傳入 record_list（或傳入 None），必須動態建立一個全新的空串列 []
# 2. 將 entry 加入該串列中並回傳
# 3. 嚴格要求：不可跨呼叫殘留舊資料，每次獨立呼叫時若未傳入 record_list，回傳長度必須為 1！

def append_to_record_safely(entry, record_list=None) -> list:
    # 請在此處實作安全預設參數模式
    if record_list is None:
        record_list = []
    record_list.append(entry)
    return record_list

# 測試用例
print("呼叫 1:", append_to_record_safely("Task 1"))
print("呼叫 2:", append_to_record_safely("Task 2")) # 不可殘留 Task 1！


In [ ]:
# 13.5.5 單元測試驗證
r1 = append_to_record_safely("A")
r2 = append_to_record_safely("B")
assert r1 == ["A"]
assert r2 == ["B"], "不可殘留前次呼叫之狀態！"
custom = ["Init"]
r3 = append_to_record_safely("C", custom)
assert r3 == ["Init", "C"]
print("🎉 13.5.5 所有測試通過！徹底掃除淺拷貝幽靈與預設參數陷阱！")


### 13.5.6 邏輯盲點五：變數遮蔽與多測資狀態殘留（考場隱形丟分元兇）

在考場中，有兩種極其隱蔽的「狀態污染」常讓學生在 OJ 上反覆吃到 WA 卻找不到原因：

#### 1. 變數遮蔽內建函式（Shadowing Built-ins）
初學者隨手將變數命名為 `sum = 0` 或 `max = 0`。在當下沒事，但當後續呼叫 `sum(my_list)` 時，直譯器會拋出 `TypeError: 'int' object is not callable`，直接毀掉整個計算邏輯！
- *命名自律*：改用 `total`, `max_val`, `min_val`, `arr`。

#### 2. 多筆測資迴圈未初始化重置（Multi-testcase State Leakage）
在處理未知行數或多組測試資料（`while True: try: ...`）時，若將答案累計變數（如 `ans = 0`、`visited = set()`、`counts = {}`）宣告在**「迴圈的外面」**，第一筆測資跑完後，`ans` 的舊數值會直接累加到第二筆測資上！
- 在自己的電腦上測第 1 筆測資：答案完全正確！
- 送到 OJ 遇到多組測資連續執行時：第一筆對、後面全錯，直接整盤 WA 0 分！
- *金律*：**「每一筆新測資的開始，所有狀態變數與容器必須強制在迴圈最開頭就地歸零初始化（Reset）！」**


In [ ]:
# 13.5.6 程式碼演示：多測資狀態未重置慘劇模擬 vs 規範化重置
import io, sys

# 模擬多筆測資串流：兩組獨立的數字清單求總和
mock_stream = '''3
10 20 30
2
5 5
'''

print("--- 壞習慣示範：累計變數放在多測資迴圈外部（狀態跨測資污染）---")
sys.stdin = io.StringIO(mock_stream)
case_id = 0
total_sum = 0 # 致命錯誤：放在 while 外面！

while True:
    try:
        line = input()
        if not line.strip():
            continue
        n = int(line)
        nums = list(map(int, input().split()))
        case_id += 1
        
        # 累加
        for x in nums:
            total_sum += x
        print(f"Case #{case_id} 計算總和: {total_sum}")
    except EOFError:
        break

print("解剖分析：Case #2 答案應該是 5+5=10，但畫面上印出了 70！因為前一筆的 60 殘留了！❌")

print("\n--- 好習慣示範：每筆測資迴圈開頭嚴格重置狀態 ---")
sys.stdin = io.StringIO(mock_stream)
case_id = 0

while True:
    try:
        line = input()
        if not line.strip():
            continue
        n = int(line)
        nums = list(map(int, input().split()))
        case_id += 1
        
        # 考場規範：在每筆測資內部就地初始化
        case_total = 0
        for x in nums:
            case_total += x
        print(f"Case #{case_id} 獨立計算總和: {case_total} ✅")
    except EOFError:
        break


### 13.5.6 語法重點回顧與核心觀念提煉

狀態清潔與防禦三大紀律：
1. **命名遠離保留字**：嚴禁使用 `sum`, `max`, `min`, `list`, `dict`, `str`, `int`, `len` 作為變數名稱。
2. **多測資重置檢查清單**：
   - 累加器變數（`total = 0`）
   - 極值初值（`best_val = -1`）
   - 造訪標記集合（`visited.clear()` 或 `visited = set()`）
   - 計數字典（`counts = {}`）
3. **自我審查習慣**：寫完多測資題目，立刻自問：「如果 OJ 連續丟入 10 組資料，我第二組的計算會不會用到第一組遺留的變數？」


In [ ]:
# 13.5.6 學生實作練習：多組測資字元去重統計器
# 任務說明：實作 process_multicase_unique_chars(lines_stream) 函式
# 傳入多行文字串流 lines_stream（每行代表一筆獨立測資）
# 對每一行文字，統計該行包含的「不重複字元個數（排除換行符號）」
# 回傳一個包含每筆測資統計結果的整數串列！
# 嚴格要求：各行之間必須完全獨立，絕不可跨行累積！

def process_multicase_unique_chars(lines_stream: str) -> list:
    results = []
    lines = lines_stream.splitlines()
    # 請在此處確保每一行的集合都在迴圈內獨立初始化
    for line in lines:
        unique_chars = set(line)
        results.append(len(unique_chars))
    return results

# 測試用例
stream = "apple\nbanana\ncat"
print("每行不重複字元統計:", process_multicase_unique_chars(stream))


In [ ]:
# 13.5.6 單元測試驗證
assert process_multicase_unique_chars("abc\naaa\n") == [3, 1]
assert process_multicase_unique_chars("hello\nworld") == [4, 5]
assert process_multicase_unique_chars("") == []
print("🎉 13.5.6 所有測試通過！徹底消滅多測資污染與變數遮蔽隱患！")


## 13.5 總結與 WA 防禦地圖

在本單元中，我們地毯式排查了無聲無息但殺傷力極大的五大語意錯誤（Logic Error）。當你上傳程式碼得到 WA 時，請按照以下地圖逐項自我檢查：

| 邏輯盲點類別 | 典型出錯特徵 | 考場防禦黃金法則 |
| :--- | :--- | :--- |
| **差一錯誤（Off-by-one）** | `range` 漏掉最後一筆、閉區間漏加一、二分搜尋漏等號 | 區間個數公理：$[a, b]$ 共有 $b - a + 1$ 個整數；二分搜尋必寫 `left <= right` |
| **運算優先級混亂** | `a + b // 2` 算錯中點、位元運算未加括號、負數平方 | **括號是免費的！** 有疑慮一律明確加上 `( )`，負數次方寫 `(-x)**2` |
| **浮點與除法陷阱** | `0.1 + 0.2 == 0.3` 失敗、負數整數除法向負無窮取整 | 浮點用容差 `abs(a-b) < 1e-9`；向零截斷用 `int(a/b)`；整數算式先乘後除 |
| **淺拷貝共享幽靈** | `[[0]*W]*H` 修改一格全網格連動、可變預設參數殘留 | 建立網格強制用列表生成式；函式預設參數一律用 `None` |
| **變數遮蔽與多測資污染** | 命名撞車 `sum=0`；第二筆測資累加到第一筆舊值 | 自律命名；所有累加器、字典與 visited 集合在每筆測資迴圈開頭就地歸零重置 |

### 🚀 下一步學習指引
消滅了語法錯誤、崩潰例外與答案錯誤之後，在 APCS 考場上還有最後一座難以逾越的高山：**「時間超限（Time Limit Exceeded, TLE）」**！
明明演算法能算對答案，但測資一擴大到 $N=10^5$，程式碼執行超過 1 秒就被評判伺服器強制切斷。
在下一單元 **13-6《時間超限（TLE）診斷：運算量估算、無窮迴圈與隱形效能坑洞防制》** 中，我們將建立每秒 $10^7$ 次操作的直覺模型，徹底剷除 `list in` 與字串串接等隱形效能黑洞！
